# Metacatalog reliability tiers

Build **cleaned** (soft exclude) and **gold** (hard include) subsets from an existing
metacatalog + LST/sources tree under `CATALOG_DIR`.

**Semantics (grill-me locked):**
- Soft exclude removes only on *positive finite* failures; missing evidence keeps the row.
- Hard include requires every gate to be *calculable* (non-NaN) and pass.
- Multi-image: `n_lst ≥ 2` **or** ≥2 bands with unique association (`n_assoc==1`; Full counts).
- Residuals: absolute Jy/beam on the **origin-band seeded LST row** only (dual RMS/mean thresh, default 1.0).
- Jitter: seed-band rematch members; fail if RMS `> 0.3 × BMAJ`.
- α corner cuts are **not** used for selection.
- Both products are first-class; contract `gold ⊆ cleaned`.

Requires `lwa-catalog[analyze]` (`healpy` for binning, `lwa-healpix` for HiPS export).

Set ``REUSE_CACHED_RELIABILITY = True`` (default) to load existing
``metacatalog_cleaned.parquet`` / ``metacatalog_gold.parquet`` instead of re-filtering.

**Run cells in order.**


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from lwa_catalog.analyze import (
    ReliabilityConfig,
    ReliabilityResult,
    filter_metacatalog_reliability,
    metacatalog_to_healpix,
    write_healpix_hips,
)
from lwa_catalog.io import read_all_lst_merged, read_metacatalog, read_table, write_table
from lwa_catalog.paths import CatalogLayout

# --- operator config ---
CATALOG_DIR = Path("/fast/claw/metacatalog_coadd2")  # existing fusion tree
NSIDE = 512
REUSE_CACHED_RELIABILITY = True  # load cleaned/gold Parquet if present; set False to regenerate
# HiPS output directory name under CATALOG_DIR; fields: {name} (full|cleaned|gold), {nside}
HIPS_DIR_TEMPLATE = "metacatalog_coadd2_{name}.hips"
CONFIG = ReliabilityConfig(
    resid_rms_thresh_jy=1.0,
    resid_mean_thresh_jy=1.0,
    jitter_bmaj_frac=0.3,
    strict=False,  # True → raise if gold ⊈ cleaned
)

layout = CatalogLayout(CATALOG_DIR)
PATHS = {
    "cleaned": layout.root / "metacatalog_cleaned.parquet",
    "gold": layout.root / "metacatalog_gold.parquet",
    "cleaned_flags": layout.root / "metacatalog_cleaned_flags.parquet",
    "gold_flags": layout.root / "metacatalog_gold_flags.parquet",
}
print("CATALOG_DIR =", layout.root.resolve())
print("REUSE_CACHED_RELIABILITY =", REUSE_CACHED_RELIABILITY)
print("HIPS_DIR_TEMPLATE =", HIPS_DIR_TEMPLATE)


CATALOG_DIR = /fast/claw/metacatalog_coadd2
REUSE_CACHED_RELIABILITY = True
HIPS_DIR_TEMPLATE = metacatalog_coadd2_{name}.hips


## Load metacatalog + LST-merged cache

In [2]:
metacatalog = read_metacatalog(layout)
print(f"metacatalog rows: {len(metacatalog)}")

_cache_ready = (
    REUSE_CACHED_RELIABILITY
    and PATHS["cleaned"].is_file()
    and PATHS["gold"].is_file()
)
if _cache_ready:
    lst_merged = None
    print("Cache hit for cleaned/gold — skipping LST-merged load")
else:
    lst_merged = read_all_lst_merged(layout)
    for band, df in lst_merged.items():
        print(f"  LST {band}: {len(df)} rows")


metacatalog rows: 51407
Cache hit for cleaned/gold — skipping LST-merged load


## Load or run cleaned + gold filters

With ``REUSE_CACHED_RELIABILITY=True`` (default), load ``metacatalog_cleaned.parquet`` /
``metacatalog_gold.parquet`` (+ flags if present). Set the flag to ``False`` to regenerate.


In [3]:
def _reliability_from_parquet(catalog_path: Path, flags_path: Path) -> ReliabilityResult:
    cat = read_table(catalog_path)
    flags = read_table(flags_path) if flags_path.is_file() else pd.DataFrame()
    if "meta_id" in cat.columns:
        meta_ids = cat["meta_id"].to_numpy(dtype=int)
    else:
        meta_ids = np.arange(len(cat), dtype=int)
    return ReliabilityResult(
        catalog=cat,
        meta_ids=meta_ids,
        tier_counts=pd.DataFrame(columns=["tier", "n_in", "n_out", "n_removed"]),
        flags=flags,
        warnings=[f"loaded from {catalog_path.name}"],
    )


_cache_ready = (
    REUSE_CACHED_RELIABILITY
    and PATHS["cleaned"].is_file()
    and PATHS["gold"].is_file()
)
if _cache_ready:
    cleaned = _reliability_from_parquet(PATHS["cleaned"], PATHS["cleaned_flags"])
    gold = _reliability_from_parquet(PATHS["gold"], PATHS["gold_flags"])
    print(f"Loaded cached cleaned={len(cleaned.catalog)}  gold={len(gold.catalog)}")
else:
    if REUSE_CACHED_RELIABILITY:
        print("Cache missing — regenerating cleaned/gold…")
    cleaned, gold = filter_metacatalog_reliability(
        metacatalog,
        layout,
        config=CONFIG,
        lst_merged=lst_merged,
    )

print("=== cleaned (soft exclude) ===")
display(cleaned.tier_counts)
print(f"n_cleaned = {len(cleaned.catalog)}")

print("\n=== gold (hard include) ===")
display(gold.tier_counts)
print(f"n_gold = {len(gold.catalog)}")

assert set(gold.meta_ids).issubset(set(cleaned.meta_ids)), "gold ⊆ cleaned violated"
print("nesting OK: gold ⊆ cleaned")

display(cleaned.catalog.head(5))
display(gold.catalog.head(5))


Loaded cached cleaned=43613  gold=41574
=== cleaned (soft exclude) ===


,tier,n_in,n_out,n_removed


n_cleaned = 43613

=== gold (hard include) ===


,tier,n_in,n_out,n_removed


n_gold = 41574
nesting OK: gold ⊆ cleaned


,origin_band,bands_present,RA,DEC,Peak_flux,Total_flux,Maj,Min,PA,DC_Maj,...,PA_Red,DC_Maj_Red,DC_Min_Red,DC_PA_Red,source_file_Red,alpha_RG,E_alpha_RG,alpha_GB,E_alpha_GB,meta_id
0,Green,"Green,Red",76.171915,38.101275,246.795584,254.274878,0.227952,0.169666,39.711654,0.037688,...,41.307458,0.056525,0.038885,99.814538,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,-0.903261,0.012825,NaN,NaN,50803
1,Blue,"Blue,Red",49.939246,41.502988,120.813431,135.007862,0.175248,0.121493,43.898105,0.068177,...,43.904245,0.074170,0.056961,96.573028,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,NaN,NaN,NaN,NaN,27777
2,Full,"Full,Blue,Green,Red",341.452704,39.687773,79.947693,91.347820,0.245133,0.177805,57.190927,0.099715,...,13.599456,0.000000,0.000000,0.000000,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,6.472693,0.258180,-6.313501,0.062575,8
3,Full,"Full,Blue,Green,Red",176.294356,19.640997,44.618883,49.868886,0.253903,0.179386,45.757153,0.106659,...,42.168981,0.171833,0.110851,46.761251,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,-0.924188,0.053393,-0.894467,0.036660,18
4,Full,"Full,Blue,Green,Red",320.923107,25.093877,40.163869,88.405575,0.309306,0.272868,40.568915,0.220063,...,43.509577,0.140971,0.128026,71.786868,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,-0.852136,0.031007,-5.180222,0.048320,25


,origin_band,bands_present,RA,DEC,Peak_flux,Total_flux,Maj,Min,PA,DC_Maj,...,PA_Red,DC_Maj_Red,DC_Min_Red,DC_PA_Red,source_file_Red,alpha_RG,E_alpha_RG,alpha_GB,E_alpha_GB,meta_id
0,Green,"Green,Red",76.171915,38.101275,246.795584,254.274878,0.227952,0.169666,39.711654,0.037688,...,41.307458,0.056525,0.038885,99.814538,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,-0.903261,0.012825,NaN,NaN,50803
1,Blue,"Blue,Red",49.939246,41.502988,120.813431,135.007862,0.175248,0.121493,43.898105,0.068177,...,43.904245,0.074170,0.056961,96.573028,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,NaN,NaN,NaN,NaN,27777
2,Full,"Full,Blue,Green,Red",341.452704,39.687773,79.947693,91.347820,0.245133,0.177805,57.190927,0.099715,...,13.599456,0.000000,0.000000,0.000000,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,6.472693,0.258180,-6.313501,0.062575,8
3,Full,"Full,Blue,Green,Red",176.294356,19.640997,44.618883,49.868886,0.253903,0.179386,45.757153,0.106659,...,42.168981,0.171833,0.110851,46.761251,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,-0.924188,0.053393,-0.894467,0.036660,18
4,Full,"Full,Blue,Green,Red",320.923107,25.093877,40.163869,88.405575,0.309306,0.272868,40.568915,0.220063,...,43.509577,0.140971,0.128026,71.786868,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,-0.852136,0.031007,-5.180222,0.048320,25


## Write subsets + companion flags (never overwrite `metacatalog.parquet`)

In [4]:
out = layout.root
paths = PATHS
assert paths["cleaned"].name != "metacatalog.parquet"

if _cache_ready:
    print("Skipping write — using cached cleaned/gold Parquet")
    for label, p in paths.items():
        status = "ok" if p.is_file() else "missing"
        print(f"  {label}: {p} ({status})")
else:
    write_table(cleaned.catalog, paths["cleaned"])
    write_table(gold.catalog, paths["gold"])
    write_table(cleaned.flags, paths["cleaned_flags"])
    write_table(gold.flags, paths["gold_flags"])
    for label, p in paths.items():
        print(f"{label}: {p} ({p.stat().st_size} bytes)")


Skipping write — using cached cleaned/gold Parquet
  cleaned: /fast/claw/metacatalog_coadd2/metacatalog_cleaned.parquet (ok)
  gold: /fast/claw/metacatalog_coadd2/metacatalog_gold.parquet (ok)
  cleaned_flags: /fast/claw/metacatalog_coadd2/metacatalog_cleaned_flags.parquet (ok)
  gold_flags: /fast/claw/metacatalog_coadd2/metacatalog_gold_flags.parquet (ok)


## Peak_flux Gaussian HEALPix → HiPS

Paints each source as an **elliptical Gaussian** on an equatorial HEALPix map
(`Peak_flux` amplitude; `Maj`/`Min` FWHM in degrees; `PA` from North toward East),
then writes a **HiPS** tile set with `lwa_healpix.healpix_to_hips` (not FITS).
Use `profile="point"` for single-pixel deposits.

Install: `pip install 'lwa-catalog[analyze]'` (`healpy` + `lwa-healpix`).

View with Aladin Lite: serve the output directory over HTTP and open `index.html`.
HiPS output directories must be empty (or new); remove old dirs before re-running.

In [5]:
# Peak_flux Gaussian HEALPix → HiPS (via lwa-healpix)
# Install: pip install 'lwa-catalog[analyze]'  (healpy + lwa-healpix)
# HiPS dirs must be empty; delete previous outputs before re-running.

out = layout.root
hips_dirs = {}
for name, cat in (
    ("full", metacatalog),
    ("cleaned", cleaned.catalog),
    ("gold", gold.catalog),
):
    m = metacatalog_to_healpix(
        cat,
        nside=NSIDE,
        weight_col="Peak_flux",
        profile="gaussian",  # Maj/Min FWHM (deg), PA N→E; use "point" for single-pixel
    )
    hips_dir = out / HIPS_DIR_TEMPLATE.format(name=name, nside=NSIDE)
    hips_dirs[name] = write_healpix_hips(
        m,
        hips_dir,
        nest=False,
        coord_frame="equatorial",
        threads=True,
        properties={"obs_title": f"metacatalog {name} (Peak_flux Gaussians)"},
    )
    print(
        f"{name}: peak_max={float(m.max()):.4g} sum={float(m.sum()):.4g} "
        f"→ {hips_dirs[name]}"
    )

print(
    "\nServe a HiPS directory over HTTP and open index.html, e.g.:\n"
    f"  python -m http.server 8000 --directory {hips_dirs['gold']}\n"
    "  then browse http://localhost:8000/"
)


full: peak_max=260.1 sum=1.02e+06 → /fast/claw/metacatalog_coadd2/metacatalog_coadd2_full.hips
cleaned: peak_max=237.4 sum=6.051e+05 → /fast/claw/metacatalog_coadd2/metacatalog_coadd2_cleaned.hips
gold: peak_max=237.4 sum=5.478e+05 → /fast/claw/metacatalog_coadd2/metacatalog_coadd2_gold.hips

Serve a HiPS directory over HTTP and open index.html, e.g.:
  python -m http.server 8000 --directory /fast/claw/metacatalog_coadd2/metacatalog_coadd2_gold.hips
  then browse http://localhost:8000/
